# 11 — Agent Observability

## Background

Debugging a failed agent run is fundamentally different from debugging a function. The agent made dozens of decisions, each conditioned on prior tool outputs. Without structured traces, you can only see the final failure — not the reasoning path that led there.

OpenTelemetry established the standard for distributed tracing in microservices. Agent observability extends the same model: every *thought*, *tool call*, and *observation* is a span; every *agent run* is a trace.

## What You'll Learn

- Span and trace data structures modeled on OpenTelemetry conventions
- Automatic instrumentation of agent reasoning loops
- Replay: reconstructing an agent run from its trace
- Anomaly detection on agent behavior (token usage spikes, tool call loops)
- Export formats: OTLP-compatible JSON, human-readable text

## Why This Matters

Without observability, agents are impossible to debug, audit, or improve systematically. Traces are also the training data for future agent improvements — you can't learn from failures you can't replay.

In [ ]:
import time, uuid, json
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional
from enum import Enum
import numpy as np

class SpanKind(str, Enum):
    THOUGHT     = 'thought'
    TOOL_CALL   = 'tool_call'
    OBSERVATION = 'observation'
    LLM_CALL    = 'llm_call'
    AGENT_RUN   = 'agent_run'

class SpanStatus(str, Enum):
    OK      = 'ok'
    ERROR   = 'error'
    TIMEOUT = 'timeout'

@dataclass
class Span:
    span_id:    str
    trace_id:   str
    parent_id:  Optional[str]
    name:       str
    kind:       SpanKind
    start_ns:   int                  # nanosecond timestamp
    end_ns:     Optional[int] = None
    status:     SpanStatus    = SpanStatus.OK
    attributes: Dict[str, Any] = field(default_factory=dict)
    events:     List[Dict]     = field(default_factory=list)

    @property
    def duration_ms(self) -> float:
        if self.end_ns is None: return 0.0
        return (self.end_ns - self.start_ns) / 1e6

    def end(self, status: SpanStatus = SpanStatus.OK, **attrs):
        self.end_ns = time.time_ns()
        self.status = status
        self.attributes.update(attrs)

    def add_event(self, name: str, **attrs):
        self.events.append({'name': name, 'ts_ns': time.time_ns(), **attrs})

    def to_dict(self) -> Dict:
        d = asdict(self)
        d['kind'] = self.kind.value
        d['status'] = self.status.value
        d['duration_ms'] = self.duration_ms
        return d

print('Span data structure defined.')


## Tracer & Trace Context

In [ ]:
from contextlib import contextmanager

class AgentTracer:
    def __init__(self):
        self._traces: Dict[str, List[Span]] = {}
        self._active_trace_id: Optional[str] = None
        self._active_span_id:  Optional[str] = None

    @contextmanager
    def start_trace(self, name: str):
        trace_id = str(uuid.uuid4())
        self._traces[trace_id] = []
        self._active_trace_id  = trace_id
        root = self._new_span(name, SpanKind.AGENT_RUN, parent_id=None)
        try:
            yield root
        finally:
            root.end()
            self._active_trace_id = None
            self._active_span_id  = None

    @contextmanager
    def start_span(self, name: str, kind: SpanKind, **attrs):
        parent_id = self._active_span_id
        span = self._new_span(name, kind, parent_id=parent_id)
        span.attributes.update(attrs)
        prev_span_id = self._active_span_id
        self._active_span_id = span.span_id
        try:
            yield span
        except Exception as e:
            span.add_event('exception', message=str(e))
            span.end(SpanStatus.ERROR, error=str(e))
            raise
        finally:
            if span.end_ns is None:
                span.end()
            self._active_span_id = prev_span_id

    def _new_span(self, name: str, kind: SpanKind, parent_id: Optional[str]) -> Span:
        span = Span(
            span_id  = str(uuid.uuid4()),
            trace_id = self._active_trace_id,
            parent_id = parent_id,
            name     = name,
            kind     = kind,
            start_ns = time.time_ns(),
        )
        if self._active_trace_id:
            self._traces[self._active_trace_id].append(span)
        return span

    def get_trace(self, trace_id: str) -> List[Span]:
        return self._traces.get(trace_id, [])

    def all_traces(self) -> Dict[str, List[Span]]:
        return dict(self._traces)

tracer = AgentTracer()
print('AgentTracer defined.')


## Instrumented ReAct Agent

In [ ]:
def mock_llm(prompt: str, system: str = '') -> tuple:
    # Returns (thought, action, action_input) or (thought, 'Final Answer', answer)
    steps = [
        ('I need to search for recent AI news.',             'web_search',  'recent AI news 2025'),
        ('Let me look at the top result in detail.',         'read_url',    'https://example.com/ai-news'),
        ('I have enough info to answer.',                    'Final Answer', 'The top AI news items are...'),
    ]
    # Cycle through steps based on how many times called
    idx = getattr(mock_llm, '_call_count', 0) % len(steps)
    mock_llm._call_count = idx + 1
    return steps[idx]

def mock_tool(name: str, input_str: str) -> str:
    time.sleep(0.005)  # simulate tool latency
    return f'[{name} result for: {input_str[:40]}]'

class InstrumentedReActAgent:
    def __init__(self, tracer: AgentTracer, max_steps: int = 10):
        self.tracer    = tracer
        self.max_steps = max_steps

    def run(self, task: str) -> Dict:
        with self.tracer.start_trace(f'agent_run:{task[:30]}') as root:
            root.attributes['task'] = task
            steps, final_answer = [], None
            scratchpad = ''

            for step_i in range(self.max_steps):
                # LLM call span
                with self.tracer.start_span(f'llm_step_{step_i}', SpanKind.LLM_CALL,
                                            step=step_i) as llm_span:
                    thought, action, action_input = mock_llm(scratchpad)
                    llm_span.attributes.update({
                        'thought':      thought,
                        'action':       action,
                        'input_tokens': len(scratchpad.split()) + 20,
                        'output_tokens': len(thought.split()) + 10,
                    })

                # Thought span
                with self.tracer.start_span(f'thought_{step_i}', SpanKind.THOUGHT,
                                            content=thought):
                    pass

                if action == 'Final Answer':
                    final_answer = action_input
                    root.add_event('final_answer', answer=final_answer)
                    break

                # Tool call span
                with self.tracer.start_span(f'tool_{action}_{step_i}', SpanKind.TOOL_CALL,
                                            tool=action, input=action_input) as tool_span:
                    try:
                        obs = mock_tool(action, action_input)
                        tool_span.attributes['output_len'] = len(obs)
                    except Exception as e:
                        obs = f'ERROR: {e}'

                # Observation span
                with self.tracer.start_span(f'obs_{step_i}', SpanKind.OBSERVATION,
                                            content=obs[:100]):
                    pass

                scratchpad += f'Thought: {thought}\nAction: {action}({action_input})\nObs: {obs}\n'
                steps.append({'thought': thought, 'action': action, 'obs': obs})

            root.attributes.update({'n_steps': len(steps), 'success': final_answer is not None})
            return {'answer': final_answer, 'steps': steps,
                    'trace_id': self.tracer._active_trace_id or root.trace_id}

agent = InstrumentedReActAgent(tracer)
result = agent.run('What are the latest AI news items?')
print(f'Answer: {result["answer"]}')
print(f'Steps:  {len(result["steps"])}')


## Trace Visualization & Export

In [ ]:
def render_trace(trace_id: str, tracer: AgentTracer):
    spans = tracer.get_trace(trace_id)
    if not spans:
        print('No spans found.'); return

    # Build parent->children map
    by_id = {s.span_id: s for s in spans}
    children: Dict[Optional[str], List[Span]] = {}
    for s in spans:
        children.setdefault(s.parent_id, []).append(s)

    def render_node(span_id: Optional[str], depth: int = 0):
        for s in children.get(span_id, []):
            indent = '  ' * depth
            status_sym = '' if s.status == SpanStatus.OK else '[ERR]'
            print(f'{indent}{status_sym}[{s.kind.value:<12}] {s.name:<40} {s.duration_ms:>7.2f}ms')
            if s.attributes.get('thought'):
                print(f'{indent}  thought: {s.attributes["thought"][:60]}')
            if s.attributes.get('tool'):
                print(f'{indent}  tool: {s.attributes["tool"]}({s.attributes.get("input","")[:30]})')
            render_node(s.span_id, depth + 1)

    root_spans = [s for s in spans if s.parent_id is None]
    for r in root_spans:
        print(f'TRACE {r.trace_id[:16]}...')
        print(f'[{r.kind.value:<12}] {r.name:<40} {r.duration_ms:>7.2f}ms')
        render_node(r.span_id, 1)

trace_id = list(tracer.all_traces().keys())[0]
render_trace(trace_id, tracer)


In [ ]:
# OTLP-compatible JSON export (subset)
def export_otlp(trace_id: str, tracer: AgentTracer) -> str:
    spans = tracer.get_trace(trace_id)
    otlp = {
        'resourceSpans': [{
            'resource': {'attributes': [{'key': 'service.name', 'value': {'stringValue': 'agent'}}]},
            'scopeSpans': [{
                'scope': {'name': 'agent-tracer', 'version': '0.1.0'},
                'spans': [{
                    'traceId':     s.trace_id,
                    'spanId':      s.span_id,
                    'parentSpanId': s.parent_id or '',
                    'name':        s.name,
                    'kind':        s.kind.value,
                    'startTimeUnixNano': str(s.start_ns),
                    'endTimeUnixNano':   str(s.end_ns or 0),
                    'status':      {'code': 1 if s.status == SpanStatus.OK else 2},
                    'attributes':  [{'key': k, 'value': {'stringValue': str(v)}}
                                    for k, v in s.attributes.items()],
                } for s in spans]
            }]
        }]
    }
    return json.dumps(otlp, indent=2)

otlp_json = export_otlp(trace_id, tracer)
print(f'OTLP export: {len(otlp_json)} bytes, {len(tracer.get_trace(trace_id))} spans')
print('First 500 chars:')
print(otlp_json[:500] + '...')


## Behavioral Anomaly Detection on Traces

Detect tool-call loops (same tool called repeatedly), token spikes, and error cascades from trace data.

In [ ]:
@dataclass
class TraceAnomaly:
    kind:    str
    detail:  str
    severity: str  # 'warning' | 'critical'

def detect_trace_anomalies(spans: List[Span]) -> List[TraceAnomaly]:
    anomalies = []

    # Tool call loop: same tool called > 3 times
    tool_counts: Dict[str, int] = {}
    for s in spans:
        if s.kind == SpanKind.TOOL_CALL:
            tool = s.attributes.get('tool', 'unknown')
            tool_counts[tool] = tool_counts.get(tool, 0) + 1
    for tool, count in tool_counts.items():
        if count > 3:
            anomalies.append(TraceAnomaly('tool_loop', f'{tool} called {count} times', 'critical'))

    # Token spike: any LLM call > 3x average
    llm_spans = [s for s in spans if s.kind == SpanKind.LLM_CALL]
    if llm_spans:
        tok_counts = [s.attributes.get('input_tokens', 0) for s in llm_spans]
        avg_tok = sum(tok_counts) / len(tok_counts)
        for s in llm_spans:
            if s.attributes.get('input_tokens', 0) > avg_tok * 3:
                anomalies.append(TraceAnomaly('token_spike',
                    f'{s.name}: {s.attributes["input_tokens"]} tokens (avg={avg_tok:.0f})', 'warning'))

    # Error cascade: > 2 error spans
    error_spans = [s for s in spans if s.status == SpanStatus.ERROR]
    if len(error_spans) > 2:
        anomalies.append(TraceAnomaly('error_cascade',
            f'{len(error_spans)} error spans in trace', 'critical'))

    return anomalies

spans   = tracer.get_trace(trace_id)
anomalies = detect_trace_anomalies(spans)
print(f'Trace has {len(spans)} spans.')
if anomalies:
    print('Anomalies detected:')
    for a in anomalies: print(f'  [{a.severity.upper()}] {a.kind}: {a.detail}')
else:
    print('No anomalies detected — trace looks healthy.')


## Key Takeaways

- Model every agent step (thought, tool call, observation, LLM call) as a span with a parent; this gives you the full call tree for free
- The OTLP-compatible export means traces slot directly into Jaeger, Tempo, or any OTel-compatible backend
- Replay is just iterating over a stored trace — no special infrastructure needed
- Tool-call loop detection catches the most common agent failure mode: getting stuck retrying the same tool
- Trace storage is cheap; the alternative (trying to reproduce a run) is expensive and often impossible